# Consistency Graph Build Status Check

This notebook verifies the health of the consistency graph SQLite database and related logs:
- `consistency_graph.sqlite` - Graph data store (nodes, edges, clusters)
- `logs/consistency.log` - Build log
- `logs/consistency_audit.jsonl` - Audit events

It also analyses build logs to identify any errors or warnings during graph generation.

In [ ]:
# Import required libraries
import json
import sqlite3
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

from scripts.consistency_graph.consistency_config import get_consistency_config
from scripts.ingest.ingest_config import IngestConfig
from scripts.utils.db_factory import get_default_vector_path, get_vector_client

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
config = IngestConfig()
consistency_config = get_consistency_config()

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

LOG_FILE = PROJECT_ROOT / "logs" / "consistency.log"
AUDIT_FILE = PROJECT_ROOT / "logs" / "consistency_audit.jsonl"
CONSISTENCY_DB = Path(consistency_config.output_sqlite)

print(f"Project root: {PROJECT_ROOT}")
print(f"Consistency graph DB: {CONSISTENCY_DB}")
print(f"Consistency graph DB exists: {CONSISTENCY_DB.exists()}")

# Check ChromaDB existence
PersistentClient_class, using_sqlite = get_vector_client(prefer="chroma")
CHROMA_PATH = Path(get_default_vector_path(Path(config.rag_data_path), using_sqlite))

print(f"rag_data_path: {config.rag_data_path}")
print(f"ChromaDB path: {CHROMA_PATH}")
print()

# Connect to ChromaDB
client = PersistentClient_class(path=str(CHROMA_PATH))

print(f"✓ Connected to ChromaDB at: {CHROMA_PATH}")
print()

## 1. Database File Check

Verify that the consistency graph database and logs exist.

In [ ]:
def check_file_status(filepath: Path, file_description: str) -> dict:
    """Check file existence, size, and modification time."""
    status = {
        "file": file_description,
        "path": str(filepath),
        "exists": filepath.exists(),
        "size_bytes": None,
        "size_mb": None,
        "modified": None,
        "age_hours": None
    }
    
    if status["exists"]:
        stat = filepath.stat()
        status["size_bytes"] = stat.st_size
        status["size_mb"] = round(stat.st_size / (1024 * 1024), 2)
        modified = datetime.fromtimestamp(stat.st_mtime)
        status["modified"] = modified.strftime("%Y-%m-%d %H:%M:%S")
        age = datetime.now() - modified
        status["age_hours"] = round(age.total_seconds() / 3600, 1)
    
    return status

# Check all files
files_to_check = [
    (CONSISTENCY_DB, "Consistency Graph SQLite DB"),
    (LOG_FILE, "Build Log"),
    (AUDIT_FILE, "Audit Log")
]

file_statuses = [check_file_status(f, desc) for f, desc in files_to_check]
df_files = pd.DataFrame(file_statuses)

print("\n" + "="*80)
print("FILE STATUS CHECK")
print("="*80)
print(df_files.to_string(index=False))
print("\n")

# Overall status
db_exists = CONSISTENCY_DB.exists()

if db_exists:
    print("✓ SUCCESS: Consistency graph database found")
else:
    print("✗ FAILURE: Consistency graph database not found")


## 2. Consistency Graph DB Analysis

Analyse the structure and contents of the consistency graph SQLite database if it exists.

In [ ]:
graph_stats = {
    "db_exists": CONSISTENCY_DB.exists(),
    "tables": [],
    "missing_tables": [],
    "nodes": 0,
    "edges": 0,
    "clusters": 0,
    "node_clusters": 0,
    "metadata_rows": 0,
    "relationships": {},
}

if CONSISTENCY_DB.exists():
    try:
        with sqlite3.connect(str(CONSISTENCY_DB)) as conn:
            cur = conn.cursor()
            tables = {
                row[0]
                for row in cur.execute(
                    "SELECT name FROM sqlite_master WHERE type='table'"
                ).fetchall()
            }
            expected_tables = {"nodes", "edges", "clusters", "node_clusters", "metadata"}
            missing_tables = sorted(expected_tables - tables)
            
            graph_stats["tables"] = sorted(tables)
            graph_stats["missing_tables"] = missing_tables
            
            print("\n" + "="*80)
            print("CONSISTENCY GRAPH DB STRUCTURE")
            print("="*80)
            
            print(f"\nTables found: {len(tables)}")
            print(f"Tables: {', '.join(sorted(tables)) if tables else 'NONE'}")
            if missing_tables:
                print(f"\n⚠ Missing tables: {', '.join(missing_tables)}")
            
            def safe_count(table_name: str) -> int:
                if table_name not in tables:
                    return 0
                return cur.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
            
            graph_stats["nodes"] = safe_count("nodes")
            graph_stats["edges"] = safe_count("edges")
            graph_stats["clusters"] = safe_count("clusters")
            graph_stats["node_clusters"] = safe_count("node_clusters")
            graph_stats["metadata_rows"] = safe_count("metadata")
            
            print("\nRow counts:")
            print(f"  Nodes: {graph_stats['nodes']}")
            print(f"  Edges: {graph_stats['edges']}")
            print(f"  Clusters: {graph_stats['clusters']}")
            print(f"  Node-Clusters: {graph_stats['node_clusters']}")
            print(f"  Metadata: {graph_stats['metadata_rows']}")
            
            if "metadata" in tables:
                metadata_rows = cur.execute(
                    "SELECT key, value, updated_at FROM metadata ORDER BY key"
                ).fetchall()
                if metadata_rows:
                    print("\nMetadata keys:")
                    for key, value, updated_at in metadata_rows:
                        print(f"  {key}: {value} (updated {updated_at})")
            
            if "edges" in tables:
                rel_rows = cur.execute(
                    "SELECT relationship, COUNT(*) FROM edges GROUP BY relationship ORDER BY COUNT(*) DESC"
                ).fetchall()
                relationships = {row[0]: row[1] for row in rel_rows}
                graph_stats["relationships"] = relationships
                if relationships:
                    print("\nEdge relationships:")
                    for rel, count in relationships.items():
                        print(f"  {rel}: {count}")
            
            print("\n✓ Consistency graph SQLite DB is accessible")
            
    except sqlite3.DatabaseError as e:
        print(f"\n✗ ERROR: Invalid SQLite database: {e}")
    except Exception as e:
        print(f"\n✗ ERROR: Failed to analyse SQLite DB: {e}")
else:
    print("\n✗ Consistency graph SQLite DB not found - cannot analyse")


## 3. SQLite Integrity Check

Verify the SQLite database integrity and foreign key consistency.

In [ ]:
if CONSISTENCY_DB.exists():
    try:
        with sqlite3.connect(str(CONSISTENCY_DB)) as conn:
            cur = conn.cursor()
            integrity_rows = cur.execute("PRAGMA integrity_check;").fetchall()
            quick_rows = cur.execute("PRAGMA quick_check;").fetchall()
            fk_rows = cur.execute("PRAGMA foreign_key_check;").fetchall()
        
        print("\n" + "="*80)
        print("SQLITE INTEGRITY CHECK")
        print("="*80)
        
        def is_ok(rows: list[tuple]) -> bool:
            return len(rows) == 1 and rows[0][0] == "ok"
        
        integrity_ok = is_ok(integrity_rows)
        quick_ok = is_ok(quick_rows)
        fk_ok = len(fk_rows) == 0
        
        print(f"\nIntegrity check: {'✓' if integrity_ok else '✗'}")
        print(f"Quick check: {'✓' if quick_ok else '✗'}")
        print(f"Foreign key check: {'✓' if fk_ok else '✗'}")
        
        if not integrity_ok:
            print("\nIntegrity check details:")
            for row in integrity_rows:
                print(f"  {row[0]}")
        
        if not quick_ok:
            print("\nQuick check details:")
            for row in quick_rows:
                print(f"  {row[0]}")
        
        if not fk_ok:
            print("\nForeign key issues:")
            for row in fk_rows[:10]:
                print(f"  {row}")
            if len(fk_rows) > 10:
                print(f"  ... {len(fk_rows) - 10} more")
        
        if integrity_ok and quick_ok and fk_ok:
            print("\n✓ SQLite integrity checks passed")
        else:
            print("\n⚠ SQLite integrity checks reported issues")
            
    except Exception as e:
        print(f"\n✗ ERROR: Failed to run SQLite checks: {e}")
else:
    print("\n✗ Consistency graph SQLite DB not found")

## 4. Build Log Analysis

Parse build logs to identify errors, warnings, and build metrics.

In [ ]:
if LOG_FILE.exists():
    try:
        with open(LOG_FILE, 'r', encoding='utf-8') as f:
            log_lines = f.readlines()
        
        print("\n" + "="*80)
        print("BUILD LOG ANALYSIS")
        print("="*80)
        
        # Parse log levels
        levels = {'ERROR': [], 'WARNING': [], 'INFO': [], 'DEBUG': []}
        for line in log_lines:
            for level in levels.keys():
                if level in line:
                    levels[level].append(line.strip())
                    break
        
        print(f"\nTotal log lines: {len(log_lines)}")
        print(f"ERROR entries: {len(levels['ERROR'])}")
        print(f"WARNING entries: {len(levels['WARNING'])}")
        print(f"INFO entries: {len(levels['INFO'])}")
        
        # Show recent errors
        if levels['ERROR']:
            print("\n❌ RECENT ERRORS (last 10):")
            for error in levels['ERROR'][-10:]:
                print(f"  {error[:200]}")
        
        # Show recent warnings
        if levels['WARNING']:
            print("\n⚠️  RECENT WARNINGS (last 5):")
            for warning in levels['WARNING'][-5:]:
                print(f"  {warning[:200]}")
        
        # Build status summary
        if levels['ERROR']:
            print("\n⚠ Build completed with ERRORS")
        elif levels['WARNING']:
            print("\n⚠ Build completed with WARNINGS")
        else:
            print("\n✓ Build completed without errors or warnings")
            
    except Exception as e:
        print(f"\n✗ ERROR: Failed to read log file: {e}")
else:
    print("\n⚠ Build log not found")

## 5. Audit Log Analysis

Parse audit events to track graph build lifecycle and identify failures.

In [ ]:
if AUDIT_FILE.exists():
    try:
        # Load audit events
        events = []
        with open(AUDIT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        events.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
        
        print("\n" + "="*80)
        print("AUDIT LOG ANALYSIS")
        print("="*80)
        
        # Filter graph build events
        graph_events = [
            e for e in events 
            if e.get('event', '').startswith('graph_')
        ]
        
        print(f"\nTotal audit events: {len(events)}")
        print(f"Graph build events: {len(graph_events)}")
        
        if graph_events:
            # Event type breakdown
            event_types = {}
            for event in graph_events:
                event_type = event.get('event', 'unknown')
                event_types[event_type] = event_types.get(event_type, 0) + 1
            
            print("\nGraph build event types:")
            for event_type, count in sorted(event_types.items()):
                print(f"  {event_type}: {count}")
            
            # Check for build completion
            completed = any(e.get('event') == 'graph_build_complete' for e in graph_events)
            errors = [e for e in graph_events if e.get('event') == 'graph_build_error']
            
            if completed:
                # Get completion details
                completion = [e for e in graph_events if e.get('event') == 'graph_build_complete'][-1]
                metadata = completion.get('metadata', {})
                print("\n✓ Build completed successfully")
                print(f"  Nodes: {metadata.get('nodes', 'N/A')}")
                print(f"  Edges: {metadata.get('edges', 'N/A')}")
                print(f"  Risk Clusters: {metadata.get('risk_clusters', 'N/A')}")
                print(f"  Topic Clusters: {metadata.get('topic_clusters', 'N/A')}")
            
            if errors:
                print(f"\n⚠ Found {len(errors)} error event(s)")
                for error in errors[-5:]:  # Show last 5
                    metadata = error.get('metadata', {})
                    print(f"  Stage: {metadata.get('stage', 'unknown')}")
                    print(f"  Error: {metadata.get('error', 'N/A')[:200]}")
                    print()
        else:
            print("\n⚠ No graph build events found in audit log")
            
    except Exception as e:
        print(f"\n✗ ERROR: Failed to read audit log: {e}")
else:
    print("\n⚠ Audit log not found")

## 6. Code Ingestion Summary

Check ChromaDB for ingested code and detected languages.

In [ ]:
if CHROMA_PATH.exists():
    try:
        import chromadb
        from collections import Counter
        
        # Initialise Chroma client
        client = chromadb.PersistentClient(path=str(CHROMA_PATH))
        
        # Get all collections
        collections = client.list_collections()
        
        print("\n" + "="*80)
        print("CODE INGESTION SUMMARY")
        print("="*80)
        
        print(f"\nChromaDB collections: {len(collections)}")
        
        if collections:
            total_docs = 0
            total_with_lang = 0
            all_languages = []
            service_types = []
            
            for collection in collections:
                try:
                    all_docs = collection.get(include=['metadatas'])
                    
                    if all_docs and all_docs['metadatas']:
                        total_docs += len(all_docs['metadatas'])
                        
                        # Extract language and service_type metadata
                        for metadata in all_docs['metadatas']:
                            if metadata:
                                lang = metadata.get('language', '').strip()
                                if lang:
                                    all_languages.append(lang)
                                    total_with_lang += 1
                                
                                svc_type = metadata.get('service_type', '').strip()
                                if svc_type:
                                    service_types.append(svc_type)
                
                except Exception as e:
                    print(f"  Warning: Error reading collection {collection.name}: {e}")
            
            print(f"\nTotal documents/chunks: {total_docs:,}")
            print(f"Documents with language metadata: {total_with_lang:,}")
            
            if all_languages:
                lang_counts = Counter(all_languages)
                print(f"\nLanguages detected: {len(lang_counts)}")
                print("\nLanguage breakdown:")
                for lang, count in sorted(lang_counts.items(), key=lambda x: x[1], reverse=True):
                    pct = (count / total_with_lang * 100) if total_with_lang > 0 else 0
                    print(f"  {lang:20s}: {count:6,} ({pct:5.1f}%)")
            
            if service_types:
                svc_counts = Counter(service_types)
                print(f"\nService types detected: {len(svc_counts)}")
                print("\nService type breakdown (top 10):")
                for svc_type, count in sorted(svc_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
                    print(f"  {svc_type:25s}: {count:6,}")
            
            # Code vs non-code detection
            code_langs = {'java', 'groovy', 'javascript', 'typescript', 'csharp', 'python', 
                         'sql', 'powershell', 'terraform', 'cs', 'js', 'ts'}
            code_docs = sum(1 for lang in all_languages if lang.lower() in code_langs)
            other_docs = total_with_lang - code_docs
            
            if total_with_lang > 0:
                print(f"\nContent type distribution:")
                print(f"  Code files       : {code_docs:6,} ({code_docs/total_with_lang*100:5.1f}%)")
                print(f"  Other/Config/Docs: {other_docs:6,} ({other_docs/total_with_lang*100:5.1f}%)")
            
            print("\n✓ Code ingestion data loaded successfully")
        else:
            print("\n⚠ No collections found in ChromaDB")
            
    except ImportError:
        print("\n⚠ chromadb not installed - cannot check ingestion data")
    except Exception as e:
        print(f"\n✗ ERROR: Failed to query ChromaDB: {e}")
else:
    print("\n⚠ ChromaDB not found at expected path")
    print(f"   Expected: {CHROMA_PATH}")


## 7. Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

# Overall status
issues = []
warnings = []
graph_stats = globals().get("graph_stats", {})

# Check for graph database
if not CONSISTENCY_DB.exists():
    issues.append("Consistency graph SQLite DB not found")
else:
    missing_tables = graph_stats.get("missing_tables", [])
    if missing_tables:
        issues.append(f"Missing tables in SQLite DB: {', '.join(missing_tables)}")
    
    nodes = graph_stats.get("nodes")
    edges = graph_stats.get("edges")
    if isinstance(nodes, int) and nodes == 0:
        warnings.append("No nodes found in SQLite DB")
    if isinstance(edges, int) and edges == 0:
        warnings.append("No edges found in SQLite DB")

# Check build log
if LOG_FILE.exists():
    with open(LOG_FILE, 'r', encoding='utf-8') as f:
        if 'ERROR' in f.read():
            issues.append("Errors found in build log")

if not issues:
    print("\n✓ Status: SUCCESS")
    print("  Consistency graph database is available")
    
    if isinstance(nodes, int) and isinstance(edges, int):
        print(f"\n  Nodes: {nodes}")
        print(f"  Edges: {edges}")
    
    print(f"\n  DB path: {CONSISTENCY_DB}")
    
    if warnings:
        print("\n  Notes:")
        for warning in warnings:
            print(f"    - {warning}")
else:
    print("\n⚠ Status: ISSUES DETECTED")
    for issue in issues:
        print(f"  - {issue}")
    
    if warnings:
        print("\n  Warnings:")
        for warning in warnings:
            print(f"  - {warning}")
    
    print("\n  Recommended actions:")
    print("  1. Check that ingestion completed successfully")
    print("  2. Review error logs above for specific failures")
    print("  3. Try rebuilding the graph:")
    print("     cd scripts/consistency_graph")
    print("     python build_consistency_graph.py")
    print("  4. Confirm the rag_data path and permissions")

print("\n" + "="*80)
